# What the ESP-Gaussian reconstruction measures

Visualise the reconstructed vorticity field, its Gaussian weights, and the environmental, internal, and full PV-gradient fields for real eddies.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import esp_pv_tools as ept
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}


In [ ]:
data = ept.load_cache()
gaussian = data[data.pv_surface_method.eq("esp_gaussian")].copy()
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
base = gaussian[gaussian.method.eq("esp_gaussian_2")].dropna(subset=["PV_grad_mag"])
examples = base.assign(rank=base.PV_grad_topo_p90_local_mag.rank(pct=True)).groupby("Cyc", group_keys=False).apply(lambda x: x.iloc[(x["rank"]-.9).abs().argsort()[:1]])

In [ ]:
for _, row in examples.iterrows():
    local = ept.local_esp_fields(row, grid, frac=2)
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), constrained_layout=True)
    panels = [("weight","viridis","Gaussian weight"),("zeta","coolwarm","Relative vorticity"),
              ("environment_mag","magma","Environmental |∇PV|"),("eddy_mag","magma","Internal |∇PV|")]
    for ax, (column, cmap, title) in zip(axes, panels):
        artist = ax.scatter(local.x, local.y, c=local[column], s=20, cmap=cmap)
        tilt.plot_ellipse(ax, row, grid, frac=1, color="cyan", lw=1.5)
        tilt.plot_ellipse(ax, row, grid, frac=2, color="white", lw=1)
        ax.set(aspect="equal", title=title, xlabel="x (km)", ylabel="y (km)")
        fig.colorbar(artist, ax=ax, shrink=.75)
    fig.suptitle(f"{row.Cyc}{int(row.Eddy)}, day {int(row.Day)}: w={row.w:.2e} s⁻¹")
    plt.show()

The cyan ellipse is the original `frac=1` core. The white ellipse is the numerical truncation at `frac=2`; cells farther away receive very small weights.